# RQ2: Pandemic Impact Analysis

**Research Question**  
Did the COVID-19 pandemic materially affect diabetes prevalence or risk factor distributions when compared to pre-pandemic years?

- H2₀: No material change in diabetes prevalence/risk during pandemic years  
- H2₁: Significant difference between pre-pandemic and pandemic-period respondents

In [ ]:
from pathlib import Path
from typing import Union, Optional
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest
import warnings
import logging

warnings.filterwarnings("ignore")

# Plotting style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
BASE_DIR = Path.cwd().parent 
PROCESSED_DATA_DIR = BASE_DIR / "data_processed"
REPORTS_DIR = BASE_DIR / "reports"
CONFIG_DIR = BASE_DIR / "config"
BRFSS_CLEAN_ZIP_FILE = PROCESSED_DATA_DIR / "BRFSS_2015_2024_cleaned.zip"

# Ensure directories exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_zipped_csv(
    zip_path: Union[str, Path],
    **read_csv_kwargs
) -> pd.DataFrame:
    """
    Read a single-CSV .zip file into a pandas DataFrame.

    Parameters
    ----------
    zip_path : str or Path
        Path to the .zip file (containing exactly one CSV, or a CSV
        with the same name as the zip).
    **read_csv_kwargs :
        Any extra keyword args passed through to pandas.read_csv
        (e.g. sep, dtype, parse_dates).

    Returns
    -------
    pd.DataFrame
    """
    zip_path = Path(zip_path)
    # For a single CSV in the zip, this is enough:
    return pd.read_csv(zip_path, compression="zip", **read_csv_kwargs)

In [ ]:
df_raw = read_zipped_csv(BRFSS_CLEAN_ZIP_FILE)
logging.info(f"Initial data loaded with {df_raw.shape[0]} rows. and columns: {df_raw.shape[1]}")

In [ ]:
def create_pandemic_periods(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create pandemic period categories:

    - Pre-Pandemic: 2015-2019
    - Pandemic: 2020-2022  
    - Post-Pandemic: 2023-2024
    """
    df = df.copy()
    
    conditions = [
        df["YEAR"].between(2015, 2019),
        df["YEAR"].between(2020, 2022),
        df["YEAR"].between(2023, 2024),
    ]
    periods = [
        "Pre-Pandemic (2015-2019)",
        "Pandemic (2020-2022)",
        "Post-Pandemic (2023-2024)",
    ]
    df["PANDEMIC_PERIOD"] = np.select(conditions, periods, default="Unknown")
    
    # Filter valid responses
    df = df[df["DIABETES"].isin([0, 1])]
    
    return df

df = create_pandemic_periods(df_raw)
df["PANDEMIC_PERIOD"].value_counts().sort_index()

In [ ]:
def calculate_prevalence_by_period(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate diabetes prevalence by pandemic period with 95% CI.
    """
    results = []
    
    for period in [
        "Pre-Pandemic (2015-2019)",
        "Pandemic (2020-2022)",
        "Post-Pandemic (2023-2024)",
    ]:
        period_data = df[df["PANDEMIC_PERIOD"] == period]
        total = len(period_data)
        diabetic = (period_data["DIABETES"] == 1).sum()
        
        prevalence = (diabetic / total * 100) if total > 0 else 0
        
        # Wilson 95% CI
        if total > 0:
            p = diabetic / total
            z = 1.96
            denominator = 1 + z**2 / total
            center = (p + z**2 / (2 * total)) / denominator
            margin = (
                z
                * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2))
                / denominator
            )
            ci_lower = max(0, (center - margin) * 100)
            ci_upper = min(100, (center + margin) * 100)
        else:
            ci_lower = ci_upper = 0
        
        results.append(
            {
                "Period": period,
                "Total_N": total,
                "Diabetic_N": diabetic,
                "Prevalence_%": prevalence,
                "CI_Lower": ci_lower,
                "CI_Upper": ci_upper,
            }
        )
    
    return pd.DataFrame(results)

period_df = calculate_prevalence_by_period(df)
period_df

In [ ]:
period_output_path = REPORTS_DIR / "rq2_prevalence_by_period.csv"
period_df.to_csv(period_output_path, index=False)
period_output_path

In [ ]:
def calculate_yearly_prevalence(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate diabetes prevalence by year with pandemic period annotation.
    """
    results = []
    
    for year in sorted(df["YEAR"].unique()):
        year_data = df[df["YEAR"] == year]
        total = len(year_data)
        diabetic = (year_data["DIABETES"] == 1).sum()
        
        prevalence = (diabetic / total * 100) if total > 0 else 0
        period = year_data["PANDEMIC_PERIOD"].iloc[0]
        
        results.append(
            {
                "Year": year,
                "Period": period,
                "Total_N": total,
                "Diabetic_N": diabetic,
                "Prevalence_%": prevalence,
            }
        )
    
    return pd.DataFrame(results)

yearly_df = calculate_yearly_prevalence(df)
yearly_df

In [ ]:
yearly_output_path = REPORTS_DIR / "rq2_yearly_prevalence.csv"
yearly_df.to_csv(yearly_output_path, index=False)
yearly_output_path

In [ ]:
def analyze_risk_factors_by_period(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze risk factor distributions across pandemic periods.
    """
    risk_factors = {
        "BMICAT": "BMI Category",
        "EXERCISE": "Exercise",
        "SMOKER": "Smoking Status",
        "HEAVY_ALCOHOL_CONSUMPTION": "Heavy Alcohol",
        "HEALTH_STATUS": "Health Status",
        "POOR_PHYSICAL_HEALTH_DAYS": "Poor Physical Health Days",
        "POOR_MENTAL_HEALTH_DAYS": "Poor Mental Health Days",
    }
    
    results = []
    
    for factor, label in risk_factors.items():
        if factor not in df.columns:
            continue
        
        for period in [
            "Pre-Pandemic (2015-2019)",
            "Pandemic (2020-2022)",
            "Post-Pandemic (2023-2024)",
        ]:
            period_data = df[df["PANDEMIC_PERIOD"] == period]
            
            if factor in [
                "BMICAT",
                "POOR_PHYSICAL_HEALTH_DAYS",
                "POOR_MENTAL_HEALTH_DAYS",
            ]:
                # Categorical/ordinal - mean/std
                mean_val = period_data[factor].mean()
                std_val = period_data[factor].std()
                results.append(
                    {
                        "Risk_Factor": label,
                        "Period": period,
                        "Mean": mean_val,
                        "Std": std_val,
                        "N": len(period_data[period_data[factor].notna()]),
                    }
                )
            else:
                # Binary - prevalence
                valid_data = period_data[period_data[factor].isin([0, 1])]
                if len(valid_data) > 0:
                    risk_present = (valid_data[factor] == 1).sum()
                    total = len(valid_data)
                    prevalence = (risk_present / total * 100) if total > 0 else 0
                    results.append(
                        {
                            "Risk_Factor": label,
                            "Period": period,
                            "Prevalence_%": prevalence,
                            "N": total,
                        }
                    )
    
    return pd.DataFrame(results)

risk_df = analyze_risk_factors_by_period(df)
risk_df.head(14)  # Show first few rows (one full risk factor)

In [ ]:
risk_output_path = REPORTS_DIR / "rq2_risk_factors_by_period.csv"
risk_df.to_csv(risk_output_path, index=False)
risk_output_path

In [ ]:
def perform_period_comparison_tests(df: pd.DataFrame) -> pd.DataFrame:
    """
    Perform two-proportion z-tests comparing diabetes prevalence between periods.
    """
    comparisons = [
        ("Pre-Pandemic (2015-2019)", "Pandemic (2020-2022)"),
        ("Pandemic (2020-2022)", "Post-Pandemic (2023-2024)"),
        ("Pre-Pandemic (2015-2019)", "Post-Pandemic (2023-2024)"),
    ]
    
    results = []
    
    for period1, period2 in comparisons:
        data1 = df[df["PANDEMIC_PERIOD"] == period1]
        data2 = df[df["PANDEMIC_PERIOD"] == period2]
        
        # Diabetes counts
        count1 = (data1["DIABETES"] == 1).sum()
        count2 = (data2["DIABETES"] == 1).sum()
        nobs1 = len(data1)
        nobs2 = len(data2)
        
        counts = np.array([count1, count2])
        nobs = np.array([nobs1, nobs2])
        
        z_stat, p_value = proportions_ztest(counts, nobs, alternative="two-sided")
        
        prev1 = (count1 / nobs1 * 100) if nobs1 > 0 else 0
        prev2 = (count2 / nobs2 * 100) if nobs2 > 0 else 0
        
        p1 = count1 / nobs1 if nobs1 > 0 else 0
        p2 = count2 / nobs2 if nobs2 > 0 else 0
        cohens_h = 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))
        
        results.append(
            {
                "Comparison": f"{period1} vs {period2}",
                "Period1": period1,
                "Period1_N": nobs1,
                "Period1_Prevalence_%": prev1,
                "Period2": period2,
                "Period2_N": nobs2,
                "Period2_Prevalence_%": prev2,
                "Difference_%": prev2 - prev1,
                "Z_Statistic": z_stat,
                "P_Value": p_value,
                "Cohens_h": cohens_h,
                "Significant_at_0.05": p_value < 0.05,
            }
        )
    
    return pd.DataFrame(results)

comparison_df = perform_period_comparison_tests(df)
comparison_df

In [ ]:
comparison_output_path = REPORTS_DIR / "rq2_statistical_comparisons.csv"
comparison_df.to_csv(comparison_output_path, index=False)
comparison_output_path

In [ ]:
def plot_pandemic_impact(
    yearly_df: pd.DataFrame,
    period_df: pd.DataFrame,
    output_path: str = None,
):
    """
    Visualize pandemic impact on diabetes prevalence.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "RQ2: COVID-19 Pandemic Impact on Diabetes Prevalence",
        fontsize=16,
        fontweight="bold",
    )
    
    # Plot 1: Yearly trend with periods
    ax1 = axes[0, 0]
    years = yearly_df["Year"]
    prevalence = yearly_df["Prevalence_%"]
    
    # Color code
    colors = []
    for period in yearly_df["Period"]:
        if "Pre-Pandemic" in period:
            colors.append("blue")
        elif "Pandemic" in period:
            colors.append("red")
        else:
            colors.append("green")
    
    ax1.plot(years, prevalence, marker="o", linewidth=2, color="black", alpha=0.6)
    ax1.scatter(years, prevalence, c=colors, s=100, alpha=0.7, edgecolors="black")
    
    # Period boundaries
    ax1.axvline(x=2019.5, color="red", linestyle="--", linewidth=2, alpha=0.5)
    ax1.axvline(x=2022.5, color="green", linestyle="--", linewidth=2, alpha=0.5)
    
    # Shaded regions
    ax1.axvspan(2015, 2019.5, alpha=0.1, color="blue", label="Pre-Pandemic")
    ax1.axvspan(2019.5, 2022.5, alpha=0.1, color="red", label="Pandemic")
    ax1.axvspan(2022.5, 2024, alpha=0.1, color="green", label="Post-Pandemic")
    
    ax1.set_xlabel("Year", fontsize=12)
    ax1.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
    ax1.set_title("Annual Prevalence with Pandemic Periods", fontsize=14, fontweight="bold")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Period comparison with CI
    ax2 = axes[0, 1]
    # Extract period labels properly (Pre-Pandemic, Pandemic, Post-Pandemic)
    periods = period_df["Period"].str.split(r"\s*\(").str[0]
    prevalence_vals = period_df["Prevalence_%"]
    ci_lower = period_df["CI_Lower"]
    ci_upper = period_df["CI_Upper"]
    x_pos = np.arange(len(periods))
    
    bars = ax2.bar(x_pos, prevalence_vals, color=["blue", "red", "green"], alpha=0.7)
    
    # Error bars
    errors = [prevalence_vals - ci_lower, ci_upper - prevalence_vals]
    ax2.errorbar(
        x_pos,
        prevalence_vals,
        yerr=errors,
        fmt="none",
        ecolor="black",
        capsize=5,
        capthick=2,
    )
    
    ax2.set_xlabel("Period", fontsize=12)
    ax2.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
    ax2.set_title("Prevalence by Pandemic Period (with 95% CI)", fontsize=14, fontweight="bold")
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(periods, rotation=15)
    ax2.grid(True, alpha=0.3, axis="y")
    
    # Add values on bars
    for i, (bar, val) in enumerate(zip(bars, prevalence_vals)):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.2,
            f"{val:.2f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
        )
    
    # Plot 3: Sample sizes
    ax3 = axes[1, 0]
    sample_sizes = period_df["Total_N"]
    ax3.bar(x_pos, sample_sizes, color=["blue", "red", "green"], alpha=0.7)
    ax3.set_xlabel("Period", fontsize=12)
    ax3.set_ylabel("Sample Size (N)", fontsize=12)
    ax3.set_title("Sample Sizes by Period", fontsize=14, fontweight="bold")
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(periods, rotation=15)
    ax3.grid(True, alpha=0.3, axis="y")
    
    # Add labels
    for i, (pos, val) in enumerate(zip(x_pos, sample_sizes)):
        ax3.text(
            pos,
            val + max(sample_sizes) * 0.01,
            f"{val:,}",
            ha="center",
            va="bottom",
            fontweight="bold",
        )
    
    # Plot 4: Year-over-year change
    ax4 = axes[1, 1]
    yoy_change = yearly_df["Prevalence_%"].diff()
    years_for_yoy = yearly_df["Year"][1:]  # Years 2016-2024 (skip first year which has no prior year)
    yoy_values = yoy_change[1:]  # Skip NaN for first year
    bar_colors = ["red" if x > 0 else "green" for x in yoy_values]
    ax4.bar(years_for_yoy, yoy_values, color=bar_colors, alpha=0.7)
    ax4.axhline(y=0, color="black", linestyle="-", linewidth=1)
    ax4.set_xlabel("Year", fontsize=12)
    ax4.set_ylabel("Year-over-Year Change (%)", fontsize=12)
    ax4.set_title("Year-over-Year Prevalence Change", fontsize=14, fontweight="bold")
    ax4.grid(True, alpha=0.3, axis="y")
    
    plt.tight_layout()
    
    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_pandemic_impact(yearly_df, period_df, REPORTS_DIR / "rq2_pandemic_impact.png")

Analysis of BRFSS data (2015–2024) reveals statistically **significant increases in diabetes prevalence across all pandemic periods**, rejecting the null hypothesis of no pandemic impact.

**Period-by-Period Comparison**
|----------|------------|--------|-------------|------|-------|-------|
|Comparison|Pre-Pandemic|Pandemic|Post-Pandemic|Change|p-value|Finding|
|----------|------------|--------|-------------|------|-------|-------|
|Pre → Pandemic|20.06%|20.40%|—|+0.34%|< 0.001|Significant ✓|
|Pandemic → Post|—|20.40%|21.12%|+0.72%|< 0.001|Significant ✓|
|Pre → Post|20.06%|—|21.12%|+1.05%|< 0.001|Significant ✓|

**Key Observations**
- **Sustained Upward Trend**: Diabetes prevalence increased monotonically across the decade, with acceleration evident during and after the pandemic period. The cumulative increase from pre-pandemic to post-pandemic (1.05%) is clinically meaningful.
- **Statistical Certainty**: All three pairwise comparisons are highly significant (p < 0.001), indicating these differences are not due to chance, despite small absolute percentage differences.
- **Effect Size Context**: While Cohen's h values are small (−0.008 to −0.026), the extremely large sample sizes (1.3M pre-pandemic, 704K pandemic, 486K post-pandemic) confer very high statistical power. Even modest differences are reliably detected.
- **Pandemic Acceleration**: The year-over-year change plots show that prevalence continued to rise during pandemic years (2020–2022), contrary to expectations of reduced healthcare utilization or diagnosis during lockdowns. The post-pandemic period (2023–2024) shows the steepest increase, suggesting a possible "catch-up" effect or increased diabetes incidence.

**Evidence supports H2₁**: The COVID-19 pandemic period was associated with **materially higher diabetes prevalence** compared to pre-pandemic years. Potential drivers include:
- **Delayed diagnoses during lockdowns** (2020–2021) that accumulated into the post-pandemic period
- **Metabolic consequences** of pandemic-related lifestyle changes (reduced activity, diet shifts, weight gain)
- **Healthcare system stress** affecting diabetes management and control
- **Pandemic-related stress and mental health impacts** on metabolic regulation

The pandemic did not reverse diabetes trends; instead, it appears to have **accelerated the pre-existing upward trend in diabetes prevalence**, particularly visible in the 2023–2024 post-pandemic period. Public health efforts should prioritize understanding the mechanisms driving this acceleration and strengthen diabetes prevention and management strategies in the post-pandemic recovery phase.

In [ ]:
def plot_statistical_comparisons(
    comparison_df: pd.DataFrame,
    output_path: str = None,
):
    """
    Visualize statistical comparison results.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "RQ2: Statistical Comparisons Between Pandemic Periods",
        fontsize=16,
        fontweight="bold",
    )
    
    comparisons = comparison_df["Comparison"].str.replace(" vs ", "\nvs\n")
    x_pos = np.arange(len(comparisons))
    
    # Plot 1: Prevalence differences
    ax1 = axes[0, 0]
    differences = comparison_df["Difference_%"]
    colors_diff = ["red" if x > 0 else "green" for x in differences]
    ax1.bar(x_pos, differences, color=colors_diff, alpha=0.7)
    ax1.axhline(y=0, color="black", linestyle="-", linewidth=2)
    ax1.set_xlabel("Comparison", fontsize=12)
    ax1.set_ylabel("Prevalence Difference (%)", fontsize=12)
    ax1.set_title("Period-to-Period Prevalence Differences", fontsize=14, fontweight="bold")
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(comparisons, fontsize=9)
    ax1.grid(True, alpha=0.3, axis="y")
    
    # Plot 2: P-values
    ax2 = axes[0, 1]
    p_values = comparison_df["P_Value"]
    colors_p = ["green" if p < 0.05 else "red" for p in p_values]
    ax2.bar(x_pos, p_values, color=colors_p, alpha=0.7)
    ax2.axhline(y=0.05, color="black", linestyle="--", linewidth=2, label="α = 0.05")
    ax2.set_xlabel("Comparison", fontsize=12)
    ax2.set_ylabel("P-Value", fontsize=12)
    ax2.set_title("Statistical Significance (Two-Proportion Z-Test)", fontsize=14, fontweight="bold")
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(comparisons, fontsize=9)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")
    
    # Plot 3: Effect sizes
    ax3 = axes[1, 0]
    cohens_h = comparison_df["Cohens_h"].abs()
    ax3.bar(x_pos, cohens_h, color="steelblue", alpha=0.7)
    ax3.axhline(y=0.2, color="orange", linestyle="--", label="Small (0.2)")
    ax3.axhline(y=0.5, color="red", linestyle="--", label="Medium (0.5)")
    ax3.set_xlabel("Comparison", fontsize=12)
    ax3.set_ylabel("|Cohen's h|", fontsize=12)
    ax3.set_title("Effect Sizes", fontsize=14, fontweight="bold")
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(comparisons, fontsize=9)
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis="y")
    
    # Plot 4: Significance summary
    ax4 = axes[1, 1]
    sig_count = comparison_df["Significant_at_0.05"].sum()
    not_sig_count = len(comparison_df) - sig_count
    labels = ["Significant\n(p < 0.05)", "Not Significant\n(p ≥ 0.05)"]
    sizes = [sig_count, not_sig_count]
    colors_pie = ["#2ecc71", "#e74c3c"]
    explode = (0.1, 0)
    ax4.pie(
        sizes,
        explode=explode,
        labels=labels,
        autopct="%1.0f%%",
        startangle=90,
        colors=colors_pie,
        textprops={"fontsize": 12},
    )
    ax4.set_title(
        f"Significance Summary\n({len(comparison_df)} comparisons)",
        fontsize=14,
        fontweight="bold",
    )
    
    plt.tight_layout()
    
    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_statistical_comparisons(comparison_df, REPORTS_DIR / "rq2_statistical_comparisons.png")


Critical Finding: 100% Statistical Significance with Paradoxically Small Effects
All three period-to-period comparisons show **p < 0.001**, yet the practical differences are modest (0.33%–1.05%). This apparent paradox reveals the interplay between statistical power and effect size in large-scale epidemiological data.

Detailed Breakdown

**1. Period-to-Period Prevalence Gaps (All Statistically Significant)**
|----------|---------------------|--------------|
|Comparison|Prevalence Difference|Interpretation|
|----------|---------------------|--------------|
|Pre-Pandemic → Pandemic|+0.33%|Modest initial increase during early pandemic (2020–2022)
|Pandemic → Post-Pandemic|+0.72%|Acceleration detected in recovery phase (2023–2024)
|Pre-Pandemic → Post-Pandemic|+1.05%|Cumulative impact: ~1 in 100 additional diabetics

The acceleration from pandemic to post-pandemic (+0.72%) is **2.2× larger** than the pre-to-pandemic shift (+0.33%), suggesting a **"catch-up" or secondary wave effect** in diabetes diagnosis/incidence post-lockdown.

**2. Why All Comparisons Are Significant Despite Small Differences**
- **Sample sizes**: 1.3M (pre-pandemic), 704K (pandemic), 486K (post-pandemic)
- **Statistical power**: Enormous sample sizes detect even trivial differences with high confidence
- **Effect sizes (Cohen's h)**: All ≤ 0.027 — below the "small effect" threshold (0.2)
- **Implication**: Differences are **real and reproducible** but **clinically modest on an individual level**; at the population level, however, even 0.7% translates to thousands of additional cases

**3. Clinical vs. Statistical Significance**
Statistical Significance: ✓ YES (all p < 0.001)
Clinical Significance: ? MODEST
  
Pre-Pandemic (2015–2019): 20.06% prevalence → ~261K diabetics
Post-Pandemic (2023–2024): 21.12% prevalence → ~271K diabetics
Net increase: ~10K additional cases (1.05% of population)

**4. Mechanism Hypothesis**
The **doubled prevalence gap** from pandemic to post-pandemic suggests:

- **Year 1–3 (2020–2022)**: Disrupted diagnosis & care → modest apparent increase (0.33%)
- **Year 4–5 (2023–2024)**: Catch-up diagnoses, metabolic consequences, healthcare system recovery → steeper increase (0.72%)

This is consistent with a **delayed-impact model** rather than a sudden pandemic shock.

In [ ]:
sig_comparisons = comparison_df["Significant_at_0.05"].sum()
total_comparisons = len(comparison_df)

logging.info("=" * 70)
logging.info("CONCLUSION")
logging.info("=" * 70)
logging.info("")
logging.info(f"• Statistical significance: {sig_comparisons}/{total_comparisons} comparisons show p < 0.05")

for _, row in comparison_df.iterrows():
    logging.info(f"\n {row['Comparison']}:")
    logging.info(f" - Difference: {row['Difference_%']:.3f}%")
    logging.info(f" - P-value: {row['P_Value']:.6f}")
    logging.info(f" - Significant: {'YES' if row['Significant_at_0.05'] else 'NO'}")

if sig_comparisons > 0:
    logging.info(f"\n✓ REJECT H2₀: Evidence of material pandemic impact on diabetes prevalence")
else:
    logging.info(f"\n✗ FAIL TO REJECT H2₀: No material pandemic impact detected")
logging.info("=" * 70)